In [9]:
%pip install -q langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [10]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-query-translation'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# Loading the Vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma   
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini")
)
print("Chunks in store:", vectorstore._collection.count())

#llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

Chunks in store: 979


In [12]:
# Decomposition Prompt (Splitting the question)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List


DECOMPOSITION_PROMPT = ChatPromptTemplate.from_template(
    "You are an AI assistant that breaks down complex questions into "
    "simpler sub-questions that can be answered independently, then "
    "combined to fully address the original question.\n\n"
    "Generate 3 sub-questions related to the input question below. "
    "Provide these sub-questions separated by newlines, with no "
    "numbering or extra commentary.\n\n"
    "Original question: {question}"
)

In [13]:
# Recursive Answer Building Prompt 
RECURSIVE_ANSWER_PROMPT = ChatPromptTemplate.from_template(
    "Here is the question you need to answer:\n\n{sub_question}\n\n"
    "Here is any available background question + answer pairs:\n\n"
    "{qa_pairs}\n\n"
    "Here is additional context relevant to the question:\n\n"
    "{context}\n\n"
    "Use the above context and any background Q&A pairs to answer "
    "the question: {sub_question}"
)

In [14]:
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStore

from rag_lab.strategies.multi_query import LineListOutputParser


class DecompositionStrategy:
    """Splits a complex query into sub-questions, answers each one
    recursively (building on prior sub-answers), then synthesizes
    a final answer from the full Q&A chain."""

    def __init__(self, vectorstore: VectorStore, llm: BaseChatModel, k: int = 4, num_subquestions: int = 3):
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        self.llm = llm
        self.num_subquestions = num_subquestions

        self.decompose_chain = DECOMPOSITION_PROMPT | llm | LineListOutputParser()
        self.recursive_chain = RECURSIVE_ANSWER_PROMPT | llm | StrOutputParser()

        self.final_synthesis_prompt = ChatPromptTemplate.from_template(
            "Here is a set of Q&A pairs that address different parts of "
            "a complex question:\n\n{qa_pairs}\n\n"
            "Use these to synthesize a final, complete answer to the "
            "original question: {question}"
        )

    def decompose(self, query: str) -> List[str]:
        return self.decompose_chain.invoke({"question": query})

    @staticmethod
    def _format_qa_pairs(qa_pairs: List[tuple[str, str]]) -> str:
        return "\n\n".join(
            f"Question: {q}\nAnswer: {a}" for q, a in qa_pairs
        )

    def retrieve(self, query: str) -> List[Document]:
        """For a decomposition strategy, `retrieve` returns the union of
        documents retrieved across all sub-questions — useful for eval
        harness inspection, even though generation doesn't use this
        flat list directly (it uses the recursive Q&A chain instead)."""
        sub_questions = self.decompose(query)
        all_docs = []
        for sub_q in sub_questions:
            all_docs.extend(self.retriever.invoke(sub_q))
        return all_docs

    def run(self, query: str) -> str:
        sub_questions = self.decompose(query)
        qa_pairs: List[tuple[str, str]] = []

        for sub_q in sub_questions:
            docs = self.retriever.invoke(sub_q)
            context = "\n\n".join(d.page_content for d in docs)
            formatted_history = self._format_qa_pairs(qa_pairs) if qa_pairs else "No prior context."

            sub_answer = self.recursive_chain.invoke({
                "sub_question": sub_q,
                "qa_pairs": formatted_history,
                "context": context,
            })
            qa_pairs.append((sub_q, sub_answer))

        final_chain = self.final_synthesis_prompt | self.llm | StrOutputParser()
        return final_chain.invoke({
            "qa_pairs": self._format_qa_pairs(qa_pairs),
            "question": query,
        })

In [16]:
# Testing
from rag_lab.strategies.decomposition import DecompositionStrategy

test_query = "How does Qian et al.'s CUDA Graph fix relate to the bottleneck MaxK-GNN identifies?"
strategy = DecompositionStrategy(vectorstore, llm=llm)

sub_qs = strategy.decompose(test_query)
print("Generated sub-questions:")
for i, q in enumerate(sub_qs, 1):
    print(f"{i}. {q}")

print("\n" + "="*60 + "\n")
print("Final synthesized answer:\n")
print(strategy.run(test_query))

Generated sub-questions:
1. What specific performance bottleneck does MaxK‑GNN identify in graph processing pipelines?
2. What is the CUDA Graph fix introduced by Qian et al., and which inefficiencies does it target?
3. How does the CUDA Graph fix directly alleviate or interact with the bottleneck highlighted by MaxK‑GNN?


Final synthesized answer:

**Short answer**

Qian et al.’s CUDA‑Graph fix directly attacks the performance limiter that MaxK‑GNN exposes: the *high‑frequency kernel‑launch and CPU‑orchestration overhead* that dominates modern GNN pipelines. By collapsing the thousands of micro‑second‑scale kernels (and the associated memory‑copy and synchronization calls) into a single CUDA‑Graph launch, the fix eliminates the per‑launch driver work, removes most host‑side scheduling, and lets the GPU drive its own data‑movement and synchronization. In other words, the fix removes exactly the bottleneck MaxK‑GNN measured to be roughly ten times larger than the actual GPU compute tim